Đọc dữ liệu

In [1]:

# CELL 2: IMPORT CAC THU VIEN

import numpy as np  # Thu vien tinh toan so hoc
import pandas as pd  # Thu vien xu ly du lieu dang bang

from pathlib import Path  # Ho tro duong dan tep

from sklearn.model_selection import KFold  # Chia du lieu thanh K fold

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa du lieu phan loai

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest

from sklearn.metrics import mean_squared_error  # Tinh MSE
from sklearn.metrics import mean_absolute_error  # Tinh MAE
from sklearn.metrics import r2_score  # Tinh R2

print("Da import thu vien thanh cong!")

# CELL 4: DOC DU LIEU THO

# Duong dan khi notebook chay trong experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Duong dan du phong khi chay tu thu muc goc du an
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Kiem tra file co ton tai khong
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file CSV")

# Doc CSV, cot dau tien la chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan dang su dung
print("Duong dan:", file_path.resolve())

# In kich thuoc du lieu
print("Kich thuoc:", diamonds.shape)

# Thong ke so dong trung lap
print("So dong trung:", diamonds.duplicated().sum())

# Thong ke so gia tri thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import thu vien thanh cong!
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc: (53943, 10)
So dong trung: 149
So gia tri thieu: 0


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: CHUAN BI DU LIEU VA K-FOLD

# Lay cac thuoc tinh dau vao, bo cot price
X = diamonds.drop(columns=["price"])

# Lay gia kim cuong lam bien muc tieu
y = diamonds["price"]

# Khoi tao bo chia 5-Fold
kf = KFold(
    n_splits=5,  # Chia thanh 5 fold
    shuffle=True,  # Tron du lieu truoc khi chia
    random_state=42  # Co dinh cach chia
)

# In thong tin
print("Tong so mau:", len(X))
print("So fold:", kf.get_n_splits())


Tong so mau: 53943
So fold: 5


Chạy 5-Fold Cross-validation

In [3]:


import time

# Khởi tạo biến lưu tổng thời gian huấn luyện
total_training_time = 0# CELL 8: HUAN LUYEN RANDOM FOREST VOI 5-FOLD

# Thu tu chat luong cua thuoc tinh color
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu chat luong cua thuoc tinh clarity
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong cua thuoc tinh cut
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot phan loai can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Tao danh sach luu ket qua cua tung fold
results = []

# Duyet qua tung lan chia cua KFold
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    # Lay cac dong Train theo chi so cua fold
    X_train = X.iloc[train_idx].copy()

    # Lay cac dong Test theo chi so cua fold
    X_test = X.iloc[test_idx].copy()

    # Lay gia thuc te cua cac mau Train
    y_train = y.iloc[train_idx]

    # Lay gia thuc te cua cac mau Test
    y_test = y.iloc[test_idx]

    # Khoi tao bo ma hoa moi cho tung fold
    encoder = OrdinalEncoder(
        categories=[
            color_order,
            clarity_order,
            cut_order
        ]
    )

    # Hoc bo ma hoa tren Train
    X_train[categorical_cols] = encoder.fit_transform(
        X_train[categorical_cols]
    )

    # Ma hoa Test bang bo ma hoa da hoc
    X_test[categorical_cols] = encoder.transform(
        X_test[categorical_cols]
    )

    # Khoi tao Random Forest voi tham so hoc mac dinh
    model = RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    )

    # Huan luyen mo hinh tren Train cua f
    # Bắt đầu tính thời gian huấn luyện Fold
    start_time = time.perf_counter()

# Huấn luyện Random Forest
    model.fit(X_train, y_train)

# Tính thời gian huấn luyện Fold
    training_time = time.perf_counter() - start_time

# Cộng thời gian vào tổng
    total_training_time += training_time

# Hiển thị thời gian huấn luyện của Fold
    print(f"Thời gian huấn luyện Fold {fold}: {training_time:.2f} giây")

    # Du doan gia tren Test cua fold
    y_pred = model.predict(X_test)

    # Tinh MSE cua fold
    mse = mean_squared_error(y_test, y_pred)

    # Tinh RMSE cua fold
    rmse = np.sqrt(mse)

    # Tinh MAE cua fold
    mae = mean_absolute_error(y_test, y_pred)

    # Tinh R2 cua fold
    r2 = r2_score(y_test, y_pred)

    # Luu ket qua vao danh sach
    results.append({
        "Fold": fold,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    # In RMSE sau moi fold
    print(f"Fold {fold}: RMSE = {rmse:.4f}")

# Chuyen ket qua thanh DataFrame
results_df = pd.DataFrame(results)

# Hien thi bang ket qua cua 5 fold
display(results_df.round(4))


Thời gian huấn luyện Fold 1: 16.58 giây
Fold 1: RMSE = 538.3778
Thời gian huấn luyện Fold 2: 11.72 giây
Fold 2: RMSE = 544.6579
Thời gian huấn luyện Fold 3: 7.10 giây
Fold 3: RMSE = 559.1233
Thời gian huấn luyện Fold 4: 8.60 giây
Fold 4: RMSE = 543.9233
Thời gian huấn luyện Fold 5: 7.71 giây
Fold 5: RMSE = 531.0911


,Fold,MSE,RMSE,MAE,R2
0,1,289850.6573,538.3778,262.9173,0.9814
1,2,296652.2799,544.6579,271.6027,0.9812
2,3,312618.8677,559.1233,269.7514,0.9806
3,4,295852.5072,543.9233,270.6857,0.9818
4,5,282057.7436,531.0911,260.7852,0.9821


Training và kết quả đánh giá

In [4]:

# CELL 10: TONG HOP KET QUA 5-FOLD

# Danh sach cac chi so can tong hop
metrics = ["MSE", "RMSE", "MAE", "R2"]

# Tinh trung binh cua tung chi so
mean_scores = results_df[metrics].mean()

# Tinh do lech chuan mau cua tung chi so
std_scores = results_df[metrics].std()

# Tao bang tong hop ket qua
summary = pd.DataFrame({
    "Chi so": metrics,
    "Trung binh": mean_scores.values,
    "Do lech chuan": std_scores.values
})

# Hien thi bang tong hop
display(summary.round(4))

# In ket qua chinh
print("MUC 1 - RANDOM FOREST 5-FOLD")
print("-" * 45)

# In RMSE trung binh
print(f"RMSE trung binh: {mean_scores['RMSE']:.4f}")

# In do lech chuan RMSE
print(f"RMSE do lech chuan: {std_scores['RMSE']:.4f}")

# In MAE trung binh
print(f"MAE trung binh: {mean_scores['MAE']:.4f}")

# In R2 trung binh
print(f"R2 trung binh: {mean_scores['R2']:.4f}")


,Chi so,Trung binh,Do lech chuan
0,MSE,295406.4111,11259.1427
1,RMSE,543.4347,10.3175
2,MAE,267.1485,4.9377
3,R2,0.9814,0.0006


MUC 1 - RANDOM FOREST 5-FOLD
---------------------------------------------
RMSE trung binh: 543.4347
RMSE do lech chuan: 10.3175
MAE trung binh: 267.1485
R2 trung binh: 0.9814
